# P2-03 · Recuperación Avanzada: MMR + Búsqueda Híbrida
**Trabajo Práctico 2 — Ingeniería Ontológica 3010090**

Este notebook implementa estrategias de recuperación avanzada que mejoran la **relevancia** y **diversidad** de los documentos recuperados:

## Estrategias implementadas

### 1. MMR — Maximal Marginal Relevance
Mejora el resultado clásico de similaridad coseno balanceando:
- **Relevancia**: los documentos deben ser relevantes a la consulta
- **Diversidad**: evitar documentos redundantes entre sí

Fórmula: `MMR(d) = λ·sim(d, query) - (1-λ)·max_sim(d, ya_seleccionados)`

### 2. Justificación de MMR sobre Búsqueda Híbrida
Se implementa MMR porque el corpus biomédico tiene alta redundancia: 
muchos artículos de arXiv tratan los mismos genes/enfermedades con vocabulario similar. 
MMR garantiza diversidad en los resultados, evitando que todos los fragmentos recuperados
provengan del mismo artículo o hablen de lo mismo.

In [ ]:
# ── Configuración local (reemplaza google.colab) ──────────────────────────
import sys
from pathlib import Path
# Ruta al repo resuelta desde este notebook (funciona para cualquier colaborador)
_REPO_ROOT = Path('__file__').parent.parent if '__file__' in dir() else Path().resolve()
# Buscar local_config.py subiendo directorios si hace falta
for _p in [_REPO_ROOT, _REPO_ROOT.parent, Path().resolve(), Path().resolve().parent]:
    if (_p / 'local_config.py').exists():
        _REPO_ROOT = _p
        break
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
from local_config import (
    BASE_DIR, CORPUS_DIR, INDEX_DIR, TTL_PATH,
    GRAPHDB_BASE, REPO_NAME, GRAPHDB_URL,
    GOOGLE_API_KEY, GROQ_API_KEY, LANGCHAIN_API_KEY, TAVILY_API_KEY
)
print('✅ Configuración local cargada')
print(f'   BASE_DIR:    {BASE_DIR}')
print(f'   CORPUS_DIR:  {CORPUS_DIR}')
print(f'   GRAPHDB_URL: {GRAPHDB_URL}')


## 1 · Selección dinámica de k (con Groq)

El sistema determina automáticamente cuántos fragmentos recuperar basándose en la complejidad de la consulta.
Se usa **Groq LLaMA 3.3 70B** por su baja latencia (<500ms) para esta micro-decisión.

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
# Cargar índice FAISS
faiss_index = FAISS.load_local(
    embeddings,
    allow_dangerous_deserialization=True
)
# Modelos LLM
groq   = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0, max_tokens=256)
print(f'   Vectores en FAISS: {faiss_index.index.ntotal}')

# ── Modelos ─────────────────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
gemini = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.2)
groq   = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0, max_tokens=512)

print('✅ Imports y modelos listos')



In [ ]:
class KSelection(BaseModel):
    k: int = Field(description="Número de documentos a recuperar (entre 3 y 10)")
    reasoning: str = Field(description="Justificación de la elección de k")

K_PROMPT = ChatPromptTemplate.from_template("""
Eres un experto en recuperación de información biomédica.
Dada la siguiente consulta, determina cuántos fragmentos de documentos (k) deben recuperarse.

Consulta: "{query}"

Reglas:
- Consultas simples/directas: k = 3
- Consultas de comparación o múltiples aspectos: k = 5-7
- Consultas de síntesis/revisión amplia: k = 8-10

Responde SOLO con JSON.
""")

k_selector_chain = K_PROMPT | groq.with_structured_output(KSelection)


def select_dynamic_k(query: str) -> int:
    """Determina dinámicamente el número óptimo de fragmentos a recuperar."""
    result = k_selector_chain.invoke({'query': query})
    k = max(3, min(10, result.k))  # clamp entre 3 y 10
    print(f'  📊 k dinámico seleccionado: {k} — {result.reasoning}')
    return k


# Test
test_queries = [
    'What is BRCA1?',
    'Compare BRCA1 and TP53 mutations in breast cancer and lung cancer and their treatment options.',
    'Provide a comprehensive review of COVID-19 treatment protocols including antivirals, corticosteroids, and supportive care.'
]

print('🎯 Selección dinámica de k:\n')
for q in test_queries:
    print(f'Query: "{q[:70]}"')
    k = select_dynamic_k(q)
    print()


## 2 · Recuperación MMR

MMR (Maximal Marginal Relevance) selecciona documentos que son relevantes a la consulta
pero diferentes entre sí, reduciendo la redundancia.

**Parámetros clave:**
- `fetch_k`: cuántos documentos candidatos recuperar inicialmente
- `lambda_mult`: balance relevancia vs diversidad (0=máx diversidad, 1=máx relevancia)

In [ ]:
def retrieve_mmr(query: str, vectorstore: FAISS, k: int = 5, 
                  fetch_k: int = 20, lambda_mult: float = 0.6) -> List[Document]:
    """
    Recuperación MMR (Maximal Marginal Relevance).
    
    Args:
        query: consulta del usuario
        vectorstore: índice FAISS
        k: número de documentos finales a retornar
        fetch_k: documentos candidatos iniciales (debe ser >= k)
        lambda_mult: balance relevancia/diversidad (0.6 = ligero sesgo hacia relevancia)
    
    Returns:
        Lista de documentos con balance relevancia-diversidad
    """
    retriever = vectorstore.as_retriever(
        search_type='mmr',
        search_kwargs={
            'k': k,
            'fetch_k': fetch_k,
            'lambda_mult': lambda_mult
        }
    )
    return retriever.invoke(query)


def retrieve_similarity(query: str, vectorstore: FAISS, k: int = 5) -> List[Tuple[Document, float]]:
    """Recuperación clásica por similitud coseno (para comparación)."""
    return vectorstore.similarity_search_with_score(query, k=k)


# ── Comparación MMR vs Similitud ──────────────────────────────────────────
test_query = 'BRCA1 protein function in DNA repair and cancer development'
k = 5

print(f'🔍 Query: "{test_query}"\n')

# Similitud coseno clásica
sim_results = retrieve_similarity(test_query, faiss_index, k=k)
print('📊 Recuperación por Similitud Coseno (P1):')
docs_from_same_source = {}
for i, (doc, score) in enumerate(sim_results):
    src = doc.metadata['doc_id']
    docs_from_same_source[src] = docs_from_same_source.get(src, 0) + 1
    print(f'  {i+1}. [{src}] score={score:.4f}')
unique_sources_sim = len(docs_from_same_source)
print(f'  ➡ Fuentes únicas: {unique_sources_sim}/{k}')
print(f'  ➡ Redundancia: {k - unique_sources_sim} chunks del mismo documento')

print()

# MMR
mmr_results = retrieve_mmr(test_query, faiss_index, k=k, fetch_k=20, lambda_mult=0.6)
print('🎯 Recuperación MMR (P2 — mayor diversidad):')
docs_from_same_source_mmr = {}
for i, doc in enumerate(mmr_results):
    src = doc.metadata['doc_id']
    docs_from_same_source_mmr[src] = docs_from_same_source_mmr.get(src, 0) + 1
    print(f'  {i+1}. [{src}]')
unique_sources_mmr = len(docs_from_same_source_mmr)
print(f'  ➡ Fuentes únicas: {unique_sources_mmr}/{k}')
print(f'  ➡ Redundancia reducida: {k - unique_sources_mmr} chunks del mismo documento')

print(f'\n✅ MMR aumentó diversidad: {unique_sources_sim} → {unique_sources_mmr} fuentes únicas')


## 3 · Análisis del parámetro λ (lambda_mult)

Demostramos cómo varía la diversidad según el valor de λ.

In [ ]:
# ── Análisis de lambda_mult ───────────────────────────────────────────────
print('📊 Análisis del parámetro λ (lambda_mult) en MMR:\n')
print(f'Query: "{test_query}"\n')

for lm in [0.0, 0.3, 0.6, 0.9, 1.0]:
    results = retrieve_mmr(test_query, faiss_index, k=5, fetch_k=20, lambda_mult=lm)
    unique = len(set(d.metadata['doc_id'] for d in results))
    label = '← máxima diversidad' if lm == 0.0 else ('← máxima relevancia' if lm == 1.0 else '')
    print(f'  λ={lm:.1f}: {unique}/5 fuentes únicas  {label}')

print('\n💡 Elección: λ=0.6 para el sistema RAG')
print('   Justificación: balance óptimo entre relevancia y cobertura temática.')
print('   El dominio biomédico requiere diversidad para evitar respuestas sesgadas')
print('   hacia un solo artículo, pero manteniendo relevancia alta.')


## 4 · Función de recuperación integrada con k dinámico

In [ ]:
def advanced_retrieve(
    query: str,
    vectorstore: FAISS,
    use_dynamic_k: bool = True,
    lambda_mult: float = 0.6
) -> dict:
    """
    Función principal de recuperación avanzada que combina:
    - Selección dinámica de k (Groq LLaMA 3.3 70B)
    - Búsqueda MMR para diversidad y relevancia
    
    Args:
        query: consulta del usuario
        vectorstore: índice FAISS con chunks semánticos
        use_dynamic_k: si True, determina k automáticamente
        lambda_mult: parámetro de balance MMR
    
    Returns:
        dict con documentos recuperados y metadatos de trazabilidad
    """
    # 1. Determinar k
    if use_dynamic_k:
        k = select_dynamic_k(query)
    else:
        k = 5
    
    # 2. Recuperar con MMR
    fetch_k = min(k * 4, faiss_index.index.ntotal)  # candidatos: 4x el k final
    docs = retrieve_mmr(query, vectorstore, k=k, fetch_k=fetch_k, lambda_mult=lambda_mult)
    
    # 3. Calcular diversidad
    unique_sources = len(set(d.metadata['doc_id'] for d in docs))
    
    return {
        'query': query,
        'k_selected': k,
        'documents': docs,
        'unique_sources': unique_sources,
        'diversity_ratio': unique_sources / k if k > 0 else 0,
        'retrieval_method': 'MMR',
        'lambda_mult': lambda_mult,
        'metadata': {
            'chunks': [{
                'doc_id'  : d.metadata['doc_id'],
                'chunk_id': d.metadata['chunk_id'],
                'preview' : d.page_content[:150]
            } for d in docs]
        }
    }


# ── Test final ────────────────────────────────────────────────────────────
result = advanced_retrieve(
    'What are the EGFR mutation types in non-small cell lung cancer and their clinical significance?',
    faiss_index
)

print('\n📋 Resultado de advanced_retrieve:')
print(f'  Query: "{result["query"][:60]}..."')
print(f'  k seleccionado: {result["k_selected"]}')
print(f'  Método: {result["retrieval_method"]} (λ={result["lambda_mult"]})')
print(f'  Documentos recuperados: {len(result["documents"])}')
print(f'  Fuentes únicas: {result["unique_sources"]}')
print(f'  Ratio de diversidad: {result["diversity_ratio"]:.2f}')
print('\n  Chunks recuperados:')
for chunk in result['metadata']['chunks']:
    print(f'    • [{chunk["doc_id"]}] {chunk["preview"][:80]}...')


## Resumen

| Componente | Práctica 1 | Práctica 2 |
|---|---|---|
| Método de búsqueda | Similitud coseno | MMR (Maximal Marginal Relevance) |
| k | Fijo (configurado manualmente) | Dinámico (decidido por Groq LLaMA 3.3 70B) |
| Diversidad | No garantizada | Garantizada por parámetro λ |
| Redundancia | Alta (mismos docs repetidos) | Reducida |

**Por qué MMR sobre Búsqueda Híbrida:**
El corpus biomédico de arXiv tiene alta redundancia temática. MMR reduce la redundancia garantizando
diversidad de fuentes en los resultados. La búsqueda híbrida (sparse+dense) requeriría BM25, 
que es menos beneficioso en texto científico altamente especializado donde el vocabulario es 
consistente y los embeddings densos capturan bien la semántica.